# Assignment 3: Milestone I Natural Language Processing
## Task 1. Basic Text Pre-processing
#### Group Name: UN_Group 5
#### Group members and ID:
* A (s)
* Tran Si Anh Khoi (s4102087)
* Tran Le Huy (s4085023)

Environment: Python 3 and Jupyter notebook

Libraries used: please include all the libraries you used in your assignment, e.g.,:
* pandas
* nltk
  * sent_tokenize
  * RegexpTokenizer
  * PorterStemmer
  * WordNetLemmatizer
* nltk data packages
  * punkt / punkt_tab
  * wordnet / omw-1.4
* itertools (chain)
* collections (Counter)

## Introduction
The objective of this task is to transform raw cosmetics and beauty product reviews into a cleaned, structured format suitable for computational analysis. With a dataset of approximately 161,200 reviews, text pre-processing is a critical first step to reduce noise and normalize the data.

This task focuses on processing the review_text feature through a multi-stage pipeline, including:
* Tokenization: Breaking down raw text into individual linguistic units using a specific regular expression.
* Normalization: Converting all text to lowercase and removing short, non-informative words.
* Stopword Removal: Filtering out common English words that do not contribute to the specific sentiment or meaning of the review.
* Frequency Filtering: Removing extremely rare words (appearing only once) and overly common words (top 20 by document frequency) to improve the quality of the resulting vocabulary.

The final outputs include a cleaned dataset "processed.csv"" and an alphabetically sorted unigram vocabulary "vocab.txt" used for sparse encoding.

## Importing libraries 

In [1]:
# Code to import libraries
import pandas as pd
from nltk import RegexpTokenizer
from nltk.tokenize import sent_tokenize
from itertools import chain
from collections import Counter
import nltk
from nltk.stem import PorterStemmer, WordNetLemmatizer
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('wordnet')
nltk.download('omw-1.4')

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\PC\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\PC\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\PC\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\PC\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


True

## 1.1 Examining and loading data
1. Extract information about the review, and then perform the pre-processing steps mentioned
below to the extracted reviews

In this initial step, we load the raw review data and the provided stopword list.

Key findings include:
* 61,284 unique review entries
* 570 words in the stopword list
* 9 missing values in the 'review_text' column

In [2]:
# Code to load the dataset and preview the first 10 rows of 'review_title'
df = pd.read_csv("cosmetics_beauty_products_reviews.csv")
print(df[['review_title']].head(10))

                          review_title
0                 Worth buying 50g one
1           Best cream to start ur day
2  perfect for summers dry for winters
3                    Not a moisturizer
4                              Average
5               not good for oily skin
6                    All time favorite
7                      "Good Product "
8                 Good eye cream combo
9                              "Olay''


In [3]:
# Code to load stopwords 
w = open('stopwords_en.txt', 'r')

# This reads the stopwords line by line, converting to lowercase and stripping whitespace then store it in a set
stopwords = set(line.strip().lower() for line in w if line.strip())

w.close()

In [4]:
# Code to examine the data and stopword list
print(f"Total reviews in dataset: {df.shape[0]}")
print(f"Total stopwords loaded: {len(stopwords)}")

Total reviews in dataset: 61284
Total stopwords loaded: 570


In [5]:
# Code Check for missing values in the review_text column
missing_reviews = df['review_text'].isnull().sum()
print(f"Missing reviews in 'review_text': {missing_reviews}")

Missing reviews in 'review_text': 9


## 1.2 Pre-processing data

### Tokenization and lowercasing
2. Tokenize each cosmetics/beauty review. The word tokenization must use the
following regular expression, r"[a-zA-Z]+(?:[-'][a-zA-Z]+)?".
3. All the words must be converted into the lower case.

In this step, we convert the raw review text into a list of individual tokens while standardizing the format. This process involves three main components: sentence segmentation, case normalization, and regex-based word tokenization.

* Lowercase Conversion: All text is converted to lowercase to ensure that the model treats "Lipstick" and "lipstick" as the same token, preventing unnecessary vocabulary inflation.
* Regex Tokenization: We use the regular expression r"[a-zA-Z]+(?:[-'][a-zA-Z]+)?".
* Sentence Segmentation: To maintain linguistic structure during tokenization, we first segment the reviews into sentences using sent_tokenize before extracting individual word tokens.
* Missing Value Handling: Since the dataset contains null values in the review_text column, the logic includes a check to ensure only valid strings are processed, while null inputs are converted into empty lists to maintain the dataframe's integrity.

In [6]:
#Code to define the tokenize function

def tokenizeReview(raw_review):
    review = raw_review.decode('utf-8') # convert the bytes-like object to python string, need this before we apply any pattern search on it
    review = review.lower() # convert all words to lowercase

    # segament into sentences
    sentences = sent_tokenize(review)

    # tokenize each sentence
    pattern = r"[a-zA-Z]+(?:[-'][a-zA-Z]+)?"
    tokenizer = RegexpTokenizer(pattern)
    token_lists = [tokenizer.tokenize(sen) for sen in sentences]

    # merge them into a list of tokens
    tokenised_review = list(chain.from_iterable(token_lists))
    return tokenised_review

In [7]:
#Code to tokenize all values in review_text column
tokens_list = [] #Create a list to store all tokens
for i in range(len(df)): #Loop through the column
    review = df['review_text'][i]
    
    #Check for valid text since there are null values in review_text column
    if isinstance(review, str):
        tokens = tokenizeReview(review.encode('utf-8')) #Apply tokenize function on all valid values
    else:
        tokens = [] #Handle null values
    
    tokens_list.append(tokens) #Store tokens into the list

df['review_text'] = tokens_list #Replace value in review_text column with the token list
df['review_text'].head()

0    [works, as, it, claims, could, see, the, diffe...
1    [it, does, what, it, claims, best, thing, is, ...
2    [i, have, been, using, this, product, for, mon...
3    [i, have, an, oily, skin, while, this, whip, a...
4    [it's, not, that, good, please, refresh, try, ...
Name: review_text, dtype: object

### 1.3 Processing data

### Lemmatization and Stemming
In this step, we reduce words to their base forms to ensure that different variations of the same word are treated as a single token. This significantly reduces the size of our vocabulary and improves the consistency of our features.
* Lemmatization: we used the WordNetLemmatizer to transform words into their dictionary base form (e.g., "was" becomes "be", "better" becomes "good"). 
* Stemming: Following lemmatization, we applied the PorterStemmer. This is a more aggressive process that chops off suffixes (e.g., "running" becomes "run", "beauty" becomes "beauti").

While lemmatization is accurate, it sometimes leaves suffixes that we want to remove for better grouping. By lemmatizing first to get the "root" and then stemming to "chop" the ends, we ensure the most compact and efficient vocabulary possible for our Cosmetics/Beauty reviews.


In [8]:
# Code to initialize Lemmatization and Stemming tools
lemmatizer = WordNetLemmatizer()
stemmer = PorterStemmer()

# Code to apply Lemmatization to get the dictionary root form of each token
df['review_text'] = [
    [lemmatizer.lemmatize(token) for token in review]
    for review in df['review_text']
]

# Code to apply Stemming to reduce words to their stems by removing suffixes
df['review_text'] = [
    [stemmer.stem(token) for token in review]
    for review in df['review_text']
]

# Code to preview the transformed tokens
print("--- After Lemmatization & Stemming ---")
print(df['review_text'].head())

--- After Lemmatization & Stemming ---
0    [work, a, it, claim, could, see, the, differ, ...
1    [it, doe, what, it, claim, best, thing, is, it...
2    [i, have, been, use, thi, product, for, month,...
3    [i, have, an, oili, skin, while, thi, whip, ac...
4    [it', not, that, good, pleas, refresh, tri, fo...
Name: review_text, dtype: object


4. Remove words with a length less than 2.
5. Remove stopwords using the provided stop words list

After normalizing the tokens, further cleaning is required to remove non-informative elements. This stage focuses on reducing the noise in the vocabulary by eliminating words that do not contribute to the sentiment or context of the beauty reviews.

* Length Filtering: Tokens with a length of less than 2 characters are removed. This eliminates single-letter artifacts (like "i", "a", or leftover punctuation) that often remain after the tokenization or stemming processes.
* Stopword Removal: Using the provided stopwords_en.txt list, we filter out high-frequency words that carry little semantic meaning (such as "the", "and", "is").
* Implementation Logic: These filters are applied after the stemming and lemmatization steps. This ensures that any word reduced to a single character or a root form by the stemmer is successfully caught and removed before the final vocabulary is generated.

In [9]:
#Remove tokens with length less than 2
df['review_text'] = [
    [w for w in review if len(w) >= 2]
    for review in df['review_text']
]

In [10]:
#Load stopword list
stopwords_list = []
with open('./stopwords_en.txt') as f:
    stopwords_xpo6 = f.read().splitlines()

In [11]:
#Filtering stop words
df['review_text'] = [
    [token for token in review if token not in stopwords_xpo6]
    for review in df['review_text']
]

6. Remove the word that appears only once in the document collection, based on term frequency.
7. Remove the top 20 most frequent words based on document frequency.

The final stage involves filtering tokens based on their distribution across the entire dataset. This ensures that the vocabulary is both manageable in size and rich in meaningful information.

* Rare Word Removal (Term Frequency): We identified and removed words that appeared only once across the entire collection of 61,284 reviews. These often consist of typos, obscure brand names, or non-informative artifacts. Removing them reduces the dimensionality of our feature space without losing significant patterns.

* Common Word Removal (Document Frequency): We calculated how many individual reviews contained each word and removed the top 20 most frequent terms. Words that appear in almost every review are usually too generic to help distinguish between different types of cosmetics or sentiments. By removing these, we allow the more descriptive, beauty-related terms to stand out in the resulting vectors.

In [12]:
#Remove the word that appears only once in the document collection, based on term frequency
all_tokens = [token for review in df['review_text'] for token in review]

term_freq = Counter(all_tokens)

words_appear_once = {word for word, count in term_freq.items() if count == 1}

df['review_text'] = [
    [token for token in review if token not in words_appear_once]
    for review in df['review_text']
]

In [13]:
#Remove the top 20 most frequent words based on document frequency
doc_freq = Counter()

for review in df['review_text']:
    unique_tokens = set(review)   # each word counted once per review
    doc_freq.update(unique_tokens)

top_20_df_words = {word for word, count in doc_freq.most_common(20)}

df['review_text'] = [
    [token for token in review if token not in top_20_df_words]
    for review in df['review_text']
]

## 1.4 Saving required outputs
Save the required output:
* processed.csv
* vocab.txt

In [ ]:
# code to save output data
df.to_csv('processed.csv', index=False)

In [15]:
#Build vocab
vocab = sorted(set(
    token for review in df['review_text'] for token in review
))
vocab_dict = {word: idx for idx, word in enumerate(vocab)}
with open('vocab.txt', 'w', encoding='utf-8') as f:
    for word, idx in vocab_dict.items():
        f.write(f"{word}:{idx}\n")

## Summary
In this task, a comprehensive text preprocessing pipeline was implemented to transform 61,284 raw beauty product reviews into a standardized, clean format suitable for mathematical representation.

### Technical Workflow:

* Tokenization & Segmentation: Used sent_tokenize for sentence splitting and a custom RegexpTokenizer with the pattern [a-zA-Z]+(?:[-'][a-zA-Z]+)? to extract unigrams while preserving hyphens and contractions.
* Normalization: Applied both WordNet Lemmatization and Porter Stemming. This dual approach ensured words were reduced to their dictionary roots before being chopped to their stems, maximizing the grouping of related word forms.
* Sequential Filtering:
  * Length Filter: Removed all tokens with fewer than 2 characters to eliminate noise.
  * Stopword Removal: Removed common English words using the provided external stopword list.
  * Term Frequency (TF) Filter: Removed words that only appeared once in the corpus to reduce the dimensionality of the feature space.
  * Document Frequency (DF) Filter: Identified and removed the top 20 most frequent words across all reviews to ensure the final vocabulary focused on beauty-related terms rather than generic language.

### Results:
The resulting vocab.txt contains 5,938 unique tokens, starting from aa and ending at zone, providing a clean foundation for the feature representations in Task 2.